In [2]:
import os
import pandas as pd
import re
import yaml  # pip install pyyaml

# === CONFIG ===
PROJECTS_DIR = r"F:\Android_Mobile_App\Initial Dataset\Config Files"
OUTPUT_DIR = r"F:\Android_Mobile_App\Initial Dataset\Analysis Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "7.1-YML_List_1st_attempt_ShallowC.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': [r'\bsdkmanager\b', r'\bavdmanager\b', r'\bemulator\b', r'\badb\b', 'connectedAndroidTest', 'connectedCheck'],
    'GitHub_GMD': ['cleanManagedDevices', 'ManagedVirtualDevice', 'managedDevices'],
    'Unit_Test': [
        'gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
        'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
    ]
}

# === CI-specific instrumentation test keywords ===
ci_instrumentation_keywords = {
    "Travis CI": [
        "connectedCheck", "connectedAndroidTest",
        "adb shell", "adb wait-for-device",
        "emulator -avd", "android-wait-for-emulator",
        r'\bsdkmanager\b', r'\bavdmanager\b', "instrumentation"
    ],
    "CircleCI": [
        "connectedCheck", "connectedAndroidTest", "circleci/android:",
        "adb shell", "emulator -avd",
        r'\bsdkmanager\b', r'\bavdmanager\b', "instrumentation"
    ],
    "GitLab CI": [
        "connectedCheck", "connectedAndroidTest",
        "adb shell am instrument", "emulator -avd",
        "instrumentation_test",
        r'\bsdkmanager\b', r'\bavdmanager\b', "instrumentation"
    ],
    "Bitrise": [
        "connectedCheck", "Start Android Emulator",
        "Wait for Android emulator",
        "adb shell am instrument",
        r'\bsdkmanager\b', r'\bavdmanager\b',
        "instrumentation"
    ]
}

# === Detect CI platform ===
def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    path = file_path.lower()
    filename = os.path.basename(path)

    if re.search(r'(^|[\\/])\.github([\\/]|$)', path) or 'github_actions' in filename:
        return "GitHub"
    if '.gitlab' in path or 'gitlab-ci' in filename:
        return "GitLab CI"
    if '.travis' in path or 'travis' in filename:
        return "Travis CI"
    if '.circleci' in path or 'circleci' in filename:
        return "CircleCI"
    if '.bitrise' in path or 'bitrise' in filename:
        return "Bitrise"

    if 'uses: actions/' in text:
        return "GitHub"
    if 'uses: docker://circleci/' in text or 'circleci/android:' in text:
        return "CircleCI"
    if 'travis' in text:
        return "Travis CI"
    if 'bitrise' in text:
        return "Bitrise"
    if 'gitlab-ci' in text:
        return "GitLab CI"

    return "Other"

# === Robust detect_testing_types ===
def detect_testing_types(yaml_text, ci_platform):
    try:
        parsed = yaml.safe_load(yaml_text)
    except Exception:
        parsed = {}

    def get_run_scripts(node):
        runs = []
        if isinstance(node, dict):
            for k, v in node.items():
                if k == 'run' and isinstance(v, str):
                    runs.append(v.lower())
                else:
                    runs.extend(get_run_scripts(v))
        elif isinstance(node, list):
            for item in node:
                runs.extend(get_run_scripts(item))
        return runs

    # === 1) Collect 'run:' scripts ===
    run_scripts = get_run_scripts(parsed)

    # === 2) Collect CI-specific top-level scripts ===
    if isinstance(parsed, dict):
        for key in ['script', 'before_script', 'before_install', 'after_script']:
            val = parsed.get(key)
            if isinstance(val, list):
                run_scripts.extend([str(v).lower() for v in val])
            elif isinstance(val, str):
                run_scripts.append(val.lower())

    run_text = '\n'.join(run_scripts)

    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    matched_keywords = []

    has_sdkmanager = bool(re.search(r'\bsdkmanager\b', run_text))
    has_avdmanager = bool(re.search(r'\bavdmanager\b', run_text))
    has_emulator = bool(re.search(r'\bemulator\b', run_text))
    has_adb = bool(re.search(r'\badb\b', run_text))
    has_connected = any(x in run_text for x in ['connectedcheck', 'connectedandroidtest'])

    github_wrappers = ['reactivecircus/android-emulator-runner']
    wrapper_used = any(wrapper in uncommented_text for wrapper in github_wrappers)

    for label, keywords in TEST_TYPES.items():
        if label == 'emulator_manual' and ci_platform in ci_instrumentation_keywords:
            continue
        if label == 'emulator_manual' and ci_platform == 'GitHub':
            continue
        for kw in keywords:
            if kw.startswith(r'\b'):
                if re.search(kw, uncommented_text):
                    found.add(label)
                    matched_keywords.append(kw)
            else:
                if kw.lower() in uncommented_text:
                    found.add(label)
                    matched_keywords.append(kw)

    if ci_platform in ci_instrumentation_keywords:
        ci_manual_cli_keywords = ci_instrumentation_keywords[ci_platform]
        cli_like = [kw for kw in ci_manual_cli_keywords if kw.startswith(r'\b') or kw in ['connectedCheck', 'connectedAndroidTest']]

        manual_flags = []
        for kw in cli_like:
            if kw == r'\bsdkmanager\b':
                if not (has_avdmanager or has_emulator or has_adb or has_connected):
                    continue
            if re.search(kw, run_text):
                manual_flags.append(kw)

        if manual_flags:
            found.add(f"{ci_platform}_emulator_manual")
            matched_keywords.extend(manual_flags)

    if ci_platform == "GitHub":
        if has_sdkmanager and (has_avdmanager or has_emulator or has_adb or has_connected):
            found.add('GitHub_emulator_manual')
            for flag, keyword in [
                (has_sdkmanager, r'\bsdkmanager\b'),
                (has_avdmanager, r'\bavdmanager\b'),
                (has_emulator, r'\bemulator\b'),
                (has_adb, r'\badb\b'),
                (has_connected, 'connectedCheck or connectedAndroidTest')
            ]:
                if flag:
                    matched_keywords.append(keyword)

    return found, matched_keywords

# === Process all YAML files ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            username = parts[0].lower() if len(parts) > 1 else 'unknown'
            project_name = parts[1].lower() if len(parts) > 2 else parts[0].lower()

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)
                    test_types, matched_keywords = detect_testing_types(raw, ci_platform)

                    is_unit_test = 'Unit_Test' in test_types
                    instr_test_types = [t for t in test_types if t != 'Unit_Test']

                    manual_types = {t for t in instr_test_types if t.endswith("_emulator_manual")}
                    hosted_types = {
                        t for t in instr_test_types
                        if "_emulator_" in t and not t.endswith("_emulator_manual")
                    }
                    if manual_types and hosted_types:
                        instr_test_types = list(hosted_types)

                    test_type_str = ', '.join(sorted(instr_test_types))
                    is_instr = bool(instr_test_types)

                    results.append({
                        'filename': filename,
                        'username': username,
                        'project': project_name,
                        'full_name': f"{username}.{project_name}",
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': is_unit_test,
                        'instrumentation_test': is_instr,
                        'matched_keywords': "; ".join(sorted(set(matched_keywords)))
                    })

                    print(f"✅ {filename} | CI: {ci_platform} | Unit: {is_unit_test} | Instr: {is_instr} | Keywords: {matched_keywords}")

            except Exception as e:
                print(f"❌ ERROR: {file_path} => {e}")
                results.append({
                    'filename': filename,
                    'username': username,
                    'project': project_name,
                    'full_name': f"{username}.{project_name}",
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False,
                    'matched_keywords': ''
                })

# === EXPORT ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ DONE! CSV generated at: {OUTPUT_CSV}")


✅ 01sadra.Detoxiom.Travis_CI..travis.yml | CI: Travis CI | Unit: False | Instr: False | Keywords: []
✅ 0GiS0.signalr-chatroom.GitHub_Actions.terraform.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0Kirby.ProgressNote.GitHub_Actions.build_check.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0Kirby.ProgressNote.GitHub_Actions.create_release.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.cboy.GitHub_Actions.android.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.cboy.GitHub_Actions.linux.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.cboy.GitHub_Actions.macos.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.cboy.GitHub_Actions.switch.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.motionmate.GitHub_Actions.android.yml | CI: GitHub | Unit: True | Instr: False | Keywords: ['test']
✅ 0xf4b1.motionmate.Travis_CI..tra